In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import timejjjjjll
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
PROJECT_DIR = Path('/content/drive/MyDrive/SWE_Project')
feature_scenarios = {
    "1_Meteo_Only": ['tmax', 'tmin', 'DOWY', 'prcp', 'vp', 'srad', 'dayl', 'elevation_m', 'longitude', 'latitude']
}

master_df = pd.read_parquet(PROJECT_DIR / "master_dataset_engineered.parquet")


class SNOTELDataset(Dataset):
    def __init__(self, df, feature_cols, target_col, seq_length=90):

        total_samples = 0
        for _, station_df in df.groupby('station'):
            if len(station_df) >= seq_length:
                total_samples += (len(station_df) - seq_length)

        # 2. Pre-allocate the memory
        num_features = len(feature_cols)
        self.X = torch.zeros((total_samples, seq_length, num_features), dtype=torch.float32)
        self.y = torch.zeros((total_samples, 1), dtype=torch.float32)

        print(f"Allocated memory for {total_samples} samples...")


        idx = 0
        for _, station_df in df.groupby('station'):
            station_df = station_df.sort_values('time')
            X_data = station_df[feature_cols].values
            y_data = station_df[target_col].values

            if len(station_df) < seq_length:
                continue

            for i in range(len(station_df) - seq_length):
                self.X[idx] = torch.tensor(X_data[i : i + seq_length], dtype=torch.float32)
                self.y[idx] = torch.tensor(y_data[i + seq_length], dtype=torch.float32)
                idx += 1

        print(f"Final Tensor Shape: {self.X.shape}")

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class SWELSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, dropout=0.3):
        super(SWELSTM, self).__init__()

        #[Batch, Sequence, Feature]
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers,
                            batch_first=True, dropout=dropout)


        self.linear = nn.Linear(hidden_dim, 1)


    def forward(self, x):
        #[batch, sequence, hidden_dim]
        out, _ = self.lstm(x)


        last_day_memory = out[:, -1, :]

        predictions = (self.linear(last_day_memory))
        return predictions

def evaluate_and_calculate(model, loader, device):
    model.eval()

    # R2 and KGE

    sum_y = 0.0
    sum_y_hat = 0.0
    sum_sq_y = 0.0
    sum_sq_y_hat = 0.0
    sum_y_y_hat = 0.0 # for correlation
    count = 0
    sum_sq_err = 0.0 # for MSE/R2 numerator

    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            preds = model(X)

            p = preds.cpu().numpy().flatten()
            t = y.cpu().numpy().flatten()

            sum_y += np.sum(t)
            sum_y_hat += np.sum(p)
            sum_sq_y += np.sum(t**2)
            sum_sq_y_hat += np.sum(p**2)
            sum_y_y_hat += np.sum(t * p)
            sum_sq_err += np.sum((t - p)**2)
            count += len(t)

    #  means
    mu_y = sum_y / count
    mu_y_hat = sum_y_pred = sum_y_hat / count

    # Standard Deviations
    var_y = (sum_sq_y / count) - (mu_y**2)
    var_y_hat = (sum_sq_y_hat / count) - (mu_y_hat**2)
    std_y = np.sqrt(max(var_y, 1e-8))
    std_y_hat = np.sqrt(max(var_y_hat, 1e-8))

    # R2
    ss_res = sum_sq_err
    ss_tot = var_y * count
    r2 = 1 - (ss_res / (ss_tot + 1e-8))

    # KGE

    covariance = (sum_y_y_hat / count) - (mu_y * mu_y_hat)
    r = covariance / (std_y * std_y_hat + 1e-8)

    # Bias (beta) and Variability (alpha)
    beta = mu_y_hat / (mu_y + 1e-8)
    alpha = std_y_hat / (std_y + 1e-8)

    kge = 1 - np.sqrt((r - 1)**2 + (beta - 1)**2 + (alpha - 1)**2)

    return r2, kge



test_stations = master_df['station'].unique()[:300]
small_df = master_df[master_df['station'].isin(test_stations)].copy()

features = feature_scenarios["1_Meteo_Only"]
target = 'WTEQ'




# Train/Val Split (Temporal)
train_df = small_df[(small_df["WY"] >= 2001) & (small_df["WY"] <= 2015)].copy()
val_df = small_df[(small_df["WY"] >= 2016) & (small_df["WY"] <= 2019)].copy()

scaler = StandardScaler()
train_df[features] = scaler.fit_transform(train_df[features])
val_df[features]   = scaler.transform(val_df[features])

#(lookback window)
train_data = SNOTELDataset(train_df, features, target, seq_length=90)
val_data = SNOTELDataset(val_df, features, target, seq_length=90)

train_loader = DataLoader(train_data, batch_size=256, shuffle=True)
val_loader = DataLoader(val_data, batch_size=256, shuffle=False)

# --- 4. THE TRAINING LOOP ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on: {device}")


model = SWELSTM(input_dim=len(features), hidden_dim=256, num_layers=2).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
epochs = 10
best_val_loss = float('inf')
for epoch in range(epochs):
    start_time = time.time()

    # -- Training Phase --
    model.train()
    train_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        predictions = model(X_batch)
        loss = criterion(predictions, y_batch)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # -- Validation Phase --
    model.eval()
    val_loss = 0.0
    with torch.no_grad(): # Turn off math gradients to save memory
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)
            val_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)


    if avg_val_loss < best_val_loss:
      best_val_loss = avg_val_loss
      torch.save(model.state_dict(), "best_model.pth")
    elapsed = time.time() - start_time

    print(f"Epoch {epoch+1:02d}/{epochs} | "
          f"Train Loss (MSE): {avg_train_loss:.4f} | "
          f"Val Loss: {avg_val_loss:.4f} | "
          f"Time: {elapsed:.1f}s")


r2, kge = evaluate_and_calculate(model, val_loader, device)
print(f"Final Model Performance | R2: {r2:.4f} | KGE: {kge:.4f}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Allocated memory for 838577 samples...
Final Tensor Shape: torch.Size([838577, 90, 10])
Allocated memory for 377339 samples...
Final Tensor Shape: torch.Size([377339, 90, 10])
Training on: cuda
Epoch 01/10 | Train Loss (MSE): 0.0091 | Val Loss: 0.0087 | Time: 188.8s
Epoch 02/10 | Train Loss (MSE): 0.0048 | Val Loss: 0.0084 | Time: 189.6s
Epoch 03/10 | Train Loss (MSE): 0.0027 | Val Loss: 0.0097 | Time: 188.2s
Epoch 04/10 | Train Loss (MSE): 0.0016 | Val Loss: 0.0085 | Time: 188.6s
Epoch 05/10 | Train Loss (MSE): 0.0010 | Val Loss: 0.0086 | Time: 188.9s
Epoch 06/10 | Train Loss (MSE): 0.0007 | Val Loss: 0.0082 | Time: 188.0s
Epoch 07/10 | Train Loss (MSE): 0.0005 | Val Loss: 0.0081 | Time: 188.4s
Epoch 08/10 | Train Loss (MSE): 0.0004 | Val Loss: 0.0084 | Time: 189.7s
Epoch 09/10 | Train Loss (MSE): 0.0003 | Val Loss: 0.0082 | Time: 189.1s
Epoch 10/10 | Train 

In [ ]:
print('hi')

hi
